# Лабораторная работа: Анализ и прогнозирование временного ряда

## Цель лабораторной работы
Изучение основных методов анализа и прогнозирование временных рядов.

## 1. Подготовка окружения и выбор датасета

Установим необходимые библиотеки и выберем набор данных для прогнозирования временного ряда. В качестве примера будем использовать данные о ежедневных продажах мороженого (Ice Cream Sales).

In [ ]:
!pip install pandas matplotlib seaborn statsmodels pmdarima sympy
# `scikit-learn` will be handled specifically for gplearn compatibility below.
!pip install arch # Для GARCH, если потребуется, но ARMA/ARIMA хватит

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
from sympy import sympify, solve, Symbol

# Создадим простой датасет для демонстрации
np.random.seed(42)
dates = pd.date_range(start='2020-01-01', periods=100, freq='D')
data = 50 + np.arange(100) * 0.5 + np.random.normal(0, 10, 100)
df = pd.DataFrame({'Date': dates, 'Sales': data})
df.set_index('Date', inplace=True)

print(df.head())

## 2. Визуализация временного ряда и его характеристик

Визуализируем сам временной ряд, а также его автокорреляционную (ACF) и частичную автокорреляционную (PACF) функции, чтобы получить представление о его поведении и определить возможные параметры для авторегрессионных моделей.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.figure(figsize=(12, 6))
plt.plot(df['Sales'])
plt.title('Ice Cream Sales Over Time')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.grid(True)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_acf(df['Sales'], lags=20, ax=axes[0])
plot_pacf(df['Sales'], lags=20, ax=axes[1])
plt.show()

## 3. Разделение временного ряда на обучающую и тестовую выборки

Для корректной оценки качества моделей необходимо разделить данные на обучающую выборку (для обучения моделей) и тестовую выборку (для оценки их производительности на ранее не виденных данных).

In [ ]:
train_size = int(len(df) * 0.8)
train_data, test_data = df[0:train_size], df[train_size:len(df)]

print(f'Размер обучающей выборки: {len(train_data)} записей')
print(f'Размер тестовой выборки: {len(test_data)} записей')

## 4. Прогнозирование временного ряда с использованием различных методов

### 4.1. Авторегрессионный метод (ARIMA)

Используем модель ARIMA (AutoRegressive Integrated Moving Average), которая является одной из наиболее популярных для прогнозирования временных рядов. Параметры $p, d, q$ будут подобраны либо вручную (по ACF/PACF), либо с помощью `pmdarima.auto_arima`.

In [ ]:
import pmdarima as pm

# Подбор оптимальных параметров ARIMA с помощью auto_arima
model_arima_auto = pm.auto_arima(train_data['Sales'], seasonal=False, suppress_warnings=True, stepwise=True,
                                 trace=True, error_action='ignore')
print(f'Оптимальные параметры ARIMA: {model_arima_auto.order}')

# Обучение модели ARIMA с найденными параметрами
model_arima = ARIMA(train_data['Sales'], order=model_arima_auto.order)
model_arima_fit = model_arima.fit()

# Прогнозирование
forecast_arima = model_arima_fit.predict(start=len(train_data), end=len(df)-1)

plt.figure(figsize=(12, 6))
plt.plot(train_data['Sales'], label='Обучающая выборка')
plt.plot(test_data['Sales'], label='Тестовая выборка')
plt.plot(forecast_arima, label='Прогноз ARIMA', linestyle='--')
plt.title('Прогноз продаж с использованием ARIMA')
plt.xlabel('Дата')
plt.ylabel('Продажи')
plt.legend()
plt.grid(True)
plt.show()

### 4.2. Метод символьной регрессии (Symbolic Regression)

Для символьной регрессии будем использовать библиотеку `gplearn` (Genetic Programming for Scikit-learn). Она позволяет находить математические выражения, которые наилучшим образом описывают данные. В данном случае, мы будем использовать её для поиска функции, зависящей от времени (или индекса).

**Примечание:** `gplearn` может потребовать немного больше времени на обучение, особенно для сложных задач.

In [ ]:
# Агрессивное удаление и установка конкретных версий для gplearn и scikit-learn
!pip uninstall -y gplearn scikit-learn joblib threadpoolctl

# Установка gplearn 0.4.0
# gplearn 0.4.0 была выпущена, когда scikit-learn версии 0.2x были актуальны.
# Попытаемся использовать scikit-learn 1.0.2, так как он более новый, но все еще в 'старой' API-зоне для gplearn 0.4.0.
# Убедитесь, что numpy и scipy не слишком новые, чтобы не вызывать конфликтов с scikit-learn 1.0.2.
# Если у вас очень новый numpy (например, 2.0.0+), scikit-learn 1.0.2 может не установиться.
# В таком случае, возможно, придется откатить numpy до 1.2x.x

!pip install gplearn==0.4.0 scikit-learn==1.0.2 joblib==1.1.1 threadpoolctl==2.2.0

# Проверим установленные версии, чтобы убедиться
!pip show gplearn
!pip show scikit-learn

In [ ]:
from gplearn.genetic import SymbolicRegressor
# from sklearn.utils import check_random_state # Эта строка может не понадобиться, но оставлена как комментарий на случай проблем

# Создаем фичи: порядковый номер дня
X_train = np.arange(len(train_data)).reshape(-1, 1)
y_train = train_data['Sales'].values

X_test = np.arange(len(train_data), len(df)).reshape(-1, 1)

# Инициализация и обучение SymbolicRegressor
est_gp = SymbolicRegressor(population_size=5000,
                            generations=20,
                            tournament_size=30,
                            stopping_criteria=0.01,
                            const_range=(-1., 1.),
                            init_depth=(2, 6),
                            init_method='half and half',
                            function_set=('add', 'sub', 'mul', 'div'),
                            metric='rmse',
                            parsimony_coefficient=0.01,
                            random_state=0,
                            n_jobs=-1, # Используем все ядра процессора
                            verbose=1)

est_gp.fit(X_train, y_train)

print(f'Найденное уравнение символьной регрессии: {est_gp._program}')

forecast_sr = est_gp.predict(X_test)

plt.figure(figsize=(12, 6))
plt.plot(train_data['Sales'], label='Обучающая выборка')
plt.plot(test_data['Sales'], label='Тестовая выборка')
plt.plot(test_data.index, forecast_sr, label='Прогноз Символьная регрессия', linestyle='--')
plt.title('Прогноз продаж с использованием Символьной регрессии')
plt.xlabel('Дата')
plt.ylabel('Продажи')
plt.legend()
plt.grid(True)
plt.show()

### 4.3. Методы на выбор из семейства МГУА (замена GMDH на аналоги)

Поскольку библиотека `gmdh` не работает на Mac, мы заменим ее на похожие подходы для моделирования нелинейных зависимостей.

Вместо методов COMBI/MULTI и MIA/RIA из GMDH, мы используем:

1. **Линейная модель (аналог COMBI/MULTI):** Простое множественное линейное регрессионное моделирование, возможно, с добавлением запаздывающих значений.
2. **Нелинейная модель (аналог MIA/RIA):** Нейронная сеть (Multi-layer Perceptron Regressor) или Polynomial Regression для захвата нелинейных зависимостей.

Эти методы позволят продемонстрировать концепции, аналогичные GMDH, но с использованием общедоступных библиотек.

#### 4.3.1. Линейный метод (аналог COMBI / MULTI)

Используем `LinearRegression` с добавлением лаговых значений (запаздывающих значений временного ряда) в качестве признаков. Это позволяет модели улавливать линейные зависимости от предыдущих значений.

In [ ]:
from sklearn.linear_model import LinearRegression

def create_lagged_features(data, lags):
    df_lagged = pd.DataFrame(data)
    for i in range(1, lags + 1):
        df_lagged[f'lag_{i}'] = df_lagged['Sales'].shift(i)
    df_lagged.dropna(inplace=True)
    return df_lagged

lags = 5 # Количество лагов для использования

df_lagged = create_lagged_features(df['Sales'], lags)

X = df_lagged.drop('Sales', axis=1)
y = df_lagged['Sales']

X_train_lin, X_test_lin, y_train_lin, y_test_lin = train_test_split(X, y, test_size=0.2, shuffle=False)

model_lin = LinearRegression()
model_lin.fit(X_train_lin, y_train_lin)

# Прогноз на тестовой выборке
forecast_lin = model_lin.predict(X_test_lin)

# Для визуализации необходимо сопоставить прогноз с тестовым индексом
forecast_lin_series = pd.Series(forecast_lin, index=y_test_lin.index)

plt.figure(figsize=(12, 6))
plt.plot(train_data['Sales'], label='Обучающая выборка')
plt.plot(test_data['Sales'], label='Тестовая выборка')
plt.plot(forecast_lin_series, label='Прогноз Линейная регрессия', linestyle='--')
plt.title('Прогноз продаж с использованием Линейной регрессии (с лагами)')
plt.xlabel('Дата')
plt.ylabel('Продажи')
plt.legend()
plt.grid(True)
plt.show()

#### 4.3.2. Нелинейный метод (аналог MIA / RIA)

Используем `MLPRegressor` (многослойный перцептрон) для моделирования нелинейных зависимостей во временном ряду. Этот метод способен улавливать сложные паттерны в данных.

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

# Для MLPRegressor лучше масштабировать данные
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
y_scaled = scaler.fit_transform(y.values.reshape(-1, 1)).flatten()

X_train_mlp, X_test_mlp, y_train_mlp, y_test_mlp = train_test_split(X_scaled, y_scaled, test_size=0.2, shuffle=False)

model_mlp = MLPRegressor(hidden_layer_sizes=(100, 50),
                           max_iter=1000,
                           random_state=42,
                           solver='adam',
                           activation='relu',
                           alpha=0.0001)

model_mlp.fit(X_train_mlp, y_train_mlp)

# Прогноз на тестовой выборке
forecast_mlp_scaled = model_mlp.predict(X_test_mlp)

# Обратное масштабирование для получения исходных значений
forecast_mlp = scaler.inverse_transform(forecast_mlp_scaled.reshape(-1, 1)).flatten()

# Для визуализации необходимо сопоставить прогноз с тестовым индексом
forecast_mlp_series = pd.Series(forecast_mlp, index=y_test_lin.index) # Используем тот же индекс, что и для линейной модели

plt.figure(figsize=(12, 6))
plt.plot(train_data['Sales'], label='Обучающая выборка')
plt.plot(test_data['Sales'], label='Тестовая выборка')
plt.plot(forecast_mlp_series, label='Прогноз MLP Regressor', linestyle='--')
plt.title('Прогноз продаж с использованием MLP Regressor')
plt.xlabel('Дата')
plt.ylabel('Продажи')
plt.legend()
plt.grid(True)
plt.show()

## 5. Оценка качества прогноза

Оценим качество прогнозов для каждой модели с использованием метрики RMSE (Root Mean Squared Error) и MAE (Mean Absolute Error).

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Сравнение ARIMA
rmse_arima = np.sqrt(mean_squared_error(test_data['Sales'], forecast_arima))
mae_arima = mean_absolute_error(test_data['Sales'], forecast_arima)
print(f'ARIMA RMSE: {rmse_arima:.2f}, MAE: {mae_arima:.2f}')

# Сравнение Символьной регрессии
rmse_sr = np.sqrt(mean_squared_error(test_data['Sales'], forecast_sr))
mae_sr = mean_absolute_error(test_data['Sales'], forecast_sr)
print(f'Symbolic Regression RMSE: {rmse_sr:.2f}, MAE: {mae_sr:.2f}')

# Сравнение Линейной регрессии (аналог COMBI/MULTI)
# Важно: здесь y_test_lin.index соответствует forecast_lin_series.index
rmse_lin = np.sqrt(mean_squared_error(y_test_lin, forecast_lin_series))
mae_lin = mean_absolute_error(y_test_lin, forecast_lin_series)
print(f'Linear Regression RMSE: {rmse_lin:.2f}, MAE: {mae_lin:.2f}')

# Сравнение MLP Regressor (аналог MIA/RIA)
# Важно: здесь y_test_lin.index соответствует forecast_mlp_series.index
rmse_mlp = np.sqrt(mean_squared_error(y_test_lin, forecast_mlp_series))
mae_mlp = mean_absolute_error(y_test_lin, forecast_mlp_series)
print(f'MLP Regressor RMSE: {rmse_mlp:.2f}, MAE: {mae_mlp:.2f}')

## Отчет

### Титульный лист

**Министерство науки и высшего образования Российской Федерации**
**Московский государственный технический университет имени Н.Э. Баумана**

**Кафедра "Системы обработки информации и управления" (ИУ5)**

**Лабораторная работа №X**
**по дисциплине "Методы прогнозирования временных рядов"**

**Тема: "Анализ и прогнозирование временного ряда"**

Выполнил: (Ваше ФИО)
Группа: ИУ5-(номер группы)

Проверил: (ФИО преподавателя)

Москва, 2025

### Описание задания

Целью данной лабораторной работы является изучение основных методов анализа и прогнозирования временных рядов. В ходе работы был выбран набор данных о ежедневных продажах мороженого. Проведены следующие шаги:

1. **Визуализация временного ряда:** Проведено построение графика временного ряда, а также автокорреляционной и частичной автокорреляционной функций для выявления основных характеристик ряда (тренды, сезонность, автокорреляция).
2. **Разделение данных:** Временной ряд был разделен на обучающую (80%) и тестовую (20%) выборки для последующей оценки моделей.
3. **Прогнозирование временного ряда:** Использованы следующие методы:
    * **ARIMA (Авторегрессионная интегрированная модель скользящего среднего):** Параметры модели $p, d, q$ были подобраны автоматически с использованием `pmdarima.auto_arima`.
    * **Символьная регрессия:** Для поиска математической функции, описывающей временной ряд, использовалась библиотека `gplearn`.
    * **Аналоги методов МГУА:** Вместо оригинальной библиотеки GMDH, недоступной на macOS, были применены следующие подходы:
        * **Линейный метод (аналог COMBI/MULTI):** Множественная линейная регрессия с использованием запаздывающих значений временного ряда в качестве признаков.
        * **Нелинейный метод (аналог MIA/RIA):** Многослойный перцептрон (`MLPRegressor`) для захвата более сложных, нелинейных зависимостей.
4. **Визуализация прогнозов:** Для каждого метода были построены графики, отображающие тестовую выборку и соответствующие прогнозы.
5. **Оценка качества:** Качество каждого прогноза было оценено с помощью метрик RMSE (Root Mean Squared Error) и MAE (Mean Absolute Error).

### Текст программы

Представлен в виде исполняемого файла `.ipynb` (Jupyter Notebook), содержащего все блоки кода и их вывод, а также подробные комментарии.

### Экранные формы с примерами выполнения программы

В Jupyter Notebook приведены следующие экранные формы:

1. **Визуализация временного ряда:** График `Ice Cream Sales Over Time`.
2. **ACF и PACF:** Графики автокорреляционной и частичной автокорреляционной функций.
3. **Прогноз ARIMA:** График `Прогноз продаж с использованием ARIMA`, показывающий обучающую, тестовую выборки и прогноз ARIMA.
4. **Прогноз Символьной регрессии:** График `Прогноз продаж с использованием Символьной регрессии`.
5. **Прогноз Линейной регрессии:** График `Прогноз продаж с использованием Линейной регрессии (с лагами)`.
6. **Прогноз MLP Regressor:** График `Прогноз продаж с использованием MLP Regressor`.
7. **Метрики качества:** Вывод значений RMSE и MAE для каждой из моделей.